# SCADA Functionality Test Visualiser

**Author:** Erick Chauke

Every grid-code functionality test leaves behind a logger spreadsheet, and right now each one is checked by eye against the acceptance procedure. This notebook is the start of doing that automatically. You paste a test export into the `data/` folder, run the notebook, and it parses the channels, lines the measured values up against their setpoints and control modes, and draws the plots that show whether the plant behaved the way each part of the NCSS SCADA Functionality Test Record (Rev 3) requires [1].

The point is that it is general. Nothing here is wired to one plant or one capture. The single config cell below is the only thing that changes from one test to the next, so the same notebook handles any site whose logger follows the standard layout. New test sections are added one at a time as the tool grows, so over time it becomes a single place to visualise and sign off the whole procedure. The first section built is the curtailment, or absolute production constraint, test.

## Setup

The cell below is the only place anything is set. To run against a new test, drop that spreadsheet into `data/` and run all cells, and nothing else is touched. It locates the workbook, records the site and time zone, sets the event windows worth highlighting, and creates `outputs/` if it is not already there. Figures are saved under `outputs/` with the source workbook name as a prefix, so results from different plants never overwrite one another.

In [2]:
# Single config cell. To point this notebook at a different test, drop the new
# spreadsheet into the data/ folder and (only if more than one file is present)
# adjust INPUT_GLOB below. Nothing else in the notebook is edited to swap sites.

from pathlib import Path

DATA_DIR = Path("data")          # input files live here (gitignored)
INPUT_GLOB = "*.xlsx"            # pattern that selects the OPC logger workbook
OUTPUT_DIR = Path("outputs")     # every figure is saved here

SITE_NAME = "Hartebeesthoek"     # plant name shown in titles
TIME_ZONE_LABEL = "UTC"         # the logger timestamps are in this zone

# Highlighted event windows, used to slice and annotate the plots.
# Each entry maps a label to a (start, end) pair of timestamps.
EVENT_WINDOWS = {
    "curtailment_35MW": ("2026-05-27 08:14:30", "2026-05-27 08:17:00"),
}

# Resolve the single input workbook without hard-coding its (confidential) name.
_candidates = sorted(DATA_DIR.glob(INPUT_GLOB))
assert len(_candidates) >= 1, f"no file matching {INPUT_GLOB} found in {DATA_DIR}"
INPUT_FILE = _candidates[0]
STEM = INPUT_FILE.stem           # used to namespace output figures per site

OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Site: {SITE_NAME} | timezone: {TIME_ZONE_LABEL}")
print(f"Input workbook resolved from {DATA_DIR}/ ({len(_candidates)} match) | stem captured")
print(f"Highlighted events: {list(EVENT_WINDOWS)}")
print(f"Figures will be written to {OUTPUT_DIR}/")

Site: Hartebeesthoek | timezone: UTC
Input workbook resolved from data/ (1 match) | stem captured
Highlighted events: ['curtailment_35MW']
Figures will be written to outputs/
